<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/notebooks/06_limpieza_ood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06b — Limpieza de conjuntos de evaluación OOD (SpaPhish + SpearPhishMX)

Este notebook armoniza y limpia los dos conjuntos reservados para la
evaluación final fuera de distribución. Se procesan en el mismo
notebook por compartir el mismo tipo de limpieza, pero se exportan
como dos archivos separados al final — nunca se combinan en un solo
conjunto, para poder evaluar por separado el desempeño en phishing
genérico (SpaPhish) frente a spear-phishing dirigido (SpearPhishMX).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Carga y armonización de esquema

SpaPhish usa columnas `subject`/`body`/`Label` (separador coma).
SpearPhishMX usa columnas `asunto`/`cuerpo_sanitizado`/`etiqueta`
(separador punto y coma). Ambos se armonizan al esquema `text`/`label`.

In [4]:
import pandas as pd

RUTA_SPAPHISH = '/content/drive/MyDrive/Spaphish dataset - DiB (1).csv'
RUTA_SPEARPHISH = '/content/drive/MyDrive/SpearPhishMX_dataset_publico (1).csv'

df_spa = pd.read_csv(RUTA_SPAPHISH)
df_spear = pd.read_csv(RUTA_SPEARPHISH, sep=';')

df_spaphish = pd.DataFrame({
    'text': (df_spa['subject'].fillna('') + ' ' + df_spa['body'].fillna('')).str.strip(),
    'label': df_spa['Label']
})

df_spearphishmx = pd.DataFrame({
    'text': (df_spear['asunto'].fillna('') + ' ' + df_spear['cuerpo_sanitizado'].fillna('')).str.strip(),
    'label': df_spear['etiqueta']
})

print('SpaPhish:', df_spaphish.shape)
print(df_spaphish['label'].value_counts())
print()
print('SpearPhishMX:', df_spearphishmx.shape)
print(df_spearphishmx['label'].value_counts())

SpaPhish: (1395, 2)
label
1    731
0    664
Name: count, dtype: int64

SpearPhishMX: (3006, 2)
label
1    1891
0    1115
Name: count, dtype: int64


## 2. Normalización de saltos de línea

Se confirmó que 51.3% de las filas de SpaPhish y 72.9% de
SpearPhishMX contienen saltos de línea (reales o literales `\n`), con
casos extremos de hasta 901 y 1,129 por fila. Se normalizan a espacio
simple, antes de cualquier otro paso, porque pueden interferir con la
detección de patrones (URLs, relleno) en los pasos siguientes.

In [23]:
def normalizar_saltos_linea(texto):
    texto = str(texto)
    texto = texto.replace('\\n', ' ')   # \n literal (2 caracteres: backslash + n)
    texto = re.sub(r'\r\n|\r|\n', ' ', texto)  # salto de línea real
    texto = re.sub(r'\s+', ' ', texto).strip()  # colapsar espacios múltiples resultantes
    return texto

df_spaphish['text'] = df_spaphish['text'].apply(normalizar_saltos_linea)
df_spearphishmx['text'] = df_spearphishmx['text'].apply(normalizar_saltos_linea)

# Verificación
def contar_saltos_linea(texto):
    texto = str(texto)
    return texto.count('\n') + texto.count('\\n')

print('SpaPhish - saltos restantes:', df_spaphish['text'].apply(contar_saltos_linea).sum())
print('SpearPhishMX - saltos restantes:', df_spearphishmx['text'].apply(contar_saltos_linea).sum())

SpaPhish - saltos restantes: 0
SpearPhishMX - saltos restantes: 0


## 3. Corrección de encoding y normalización de caracteres

Corrección de mojibake, homóglifos usados como disfraz visual, y
marcas combinadoras invisibles — mismo procedimiento validado en el
notebook 03.
Se detectaron dos patrones nuevos en la auditoría manual, no cubiertos por la limpieza anterior:

Símbolos musicales Unicode usados como marcas combinadoras (Y𝅳our 𝅺acc𝅸o𝅴unt𝅹 𝅹ha𝅹s been 𝅸di𝅸sabled), rango U+1D165–U+1D17A.
Letras latinas con acento (Í, É, Á...) insertadas como separador visible entre cada letra de una palabra (SÍuÍbÍsÍcÍrÍiÍpÍtÍiÍoÍnÍ), una técnica distinta a las marcas combinadoras invisibles: aquí el carácter insertado es una letra real y visible, no invisible.

In [18]:
!pip install ftfy unidecode -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 11.6 MB/s eta 0:00:00


In [24]:
COMBINING_MARK_RE = re.compile(
    r"[\u0300-\u036f\u0483-\u0489\u0591-\u05c7\u0610-\u061a\u064b-\u065f"
    r"\u0670\u06d6-\u06dc\u0e31\u0e34-\u0e3a\u0e47-\u0e4e\u20d0-\u20ff\ufe00-\ufe0f"
    r"\U0001D165-\U0001D17A]"
)

PATRON_LETRA_INTERCALADA = re.compile(
    r'(?:[A-Za-z][ÁÉÍÓÚáéíóúÑñ]){4,}[A-Za-z]?'
)

def limpiar_letra_intercalada(texto):
    def _quitar_intercalado(m):
        bloque = m.group(0)
        for vocal in 'ÁÉÍÓÚáéíóúÑñ':
            bloque = bloque.replace(vocal, '')
        return bloque
    return PATRON_LETRA_INTERCALADA.sub(_quitar_intercalado, texto)


def corregir_encoding_y_caracteres(texto):
    if not isinstance(texto, str):
        return ""
    texto = ftfy.fix_text(texto)
    texto = html.unescape(texto)
    texto = LOOKALIKE_SCRIPT_RE.sub(lambda m: unidecode(m.group(0)), texto)
    texto = COMBINING_MARK_RE.sub("", texto)
    texto = INVISIBLE_CHARS_RE.sub("", texto)
    texto = limpiar_letra_intercalada(texto)
    texto = unicodedata.normalize("NFKC", texto)
    return texto


df_spaphish['text'] = df_spaphish['text'].apply(corregir_encoding_y_caracteres)
df_spearphishmx['text'] = df_spearphishmx['text'].apply(corregir_encoding_y_caracteres)

print('Corrección de ofuscación adicional (símbolos musicales, letra intercalada) aplicada.')

Corrección de ofuscación adicional (símbolos musicales, letra intercalada) aplicada.


## 4. Verificación de duplicados

Se confirmó en el diagnóstico inicial que ambos conjuntos ya vienen
prácticamente sin duplicados (0% en SpaPhish, 0.03% en SpearPhishMX).
Se verifica de nuevo sobre el texto ya limpio, por si la limpieza
reveló algún duplicado oculto.

In [25]:
n_dup_spa = df_spaphish.duplicated(subset=['text']).sum()
n_dup_spear = df_spearphishmx.duplicated(subset=['text']).sum()

print(f'SpaPhish - duplicados tras limpieza: {n_dup_spa}')
print(f'SpearPhishMX - duplicados tras limpieza: {n_dup_spear}')

df_spaphish = df_spaphish.drop_duplicates(subset=['text']).reset_index(drop=True)
df_spearphishmx = df_spearphishmx.drop_duplicates(subset=['text']).reset_index(drop=True)

print()
print('Filas finales SpaPhish:', len(df_spaphish))
print('Filas finales SpearPhishMX:', len(df_spearphishmx))

SpaPhish - duplicados tras limpieza: 0
SpearPhishMX - duplicados tras limpieza: 0

Filas finales SpaPhish: 1394
Filas finales SpearPhishMX: 3005


## 4b. Revisión de muestra aleatoria antes de exportar

In [26]:
muestra_spa = df_spaphish.sample(50, random_state=7)

for i, row in muestra_spa.iterrows():
    print(f"=== SpaPhish - Fila {i} | label: {row['label']} ===")
    print(row['text'])
    print()
    print("-" * 100)
    print()

=== SpaPhish - Fila 758 | label: 0 ===
Alcance al mensaje sobre el código de conducta Ciudad de Plata, Veracruz a 25 de septiembre de 2020. Estimada comunidad de DataCore En alcance al mensaje anterior referido al código de conducta de DataCore, deseo comunicarles que esto es parte del proceso de normalización y restablecimiento de procedimientos, manuales, y protocolos que nos marca la secretaría de la función pública. Este tipo de códigos, manuales, y procedimientos que poco a poco vamos a ir restableciendo o creando cuando sea necesario, serán muy útiles para el correcto funcionamiento del DataCore. Les pido entonces que lo lean y llenen la carta compromiso que se adjuntó. Saludos, Alberto Dr. Alberto M. D. Director General Interino DataCore

----------------------------------------------------------------------------------------------------

=== SpaPhish - Fila 902 | label: 0 ===
Descarga C.V. COVID liga archivo=>https://cvcovid.salud.gob.mx/comprobanteVacunacion/pdf?md5=AQIBovmJ-y

In [27]:
muestra_spear = df_spearphishmx.sample(50, random_state=7)

for i, row in muestra_spear.iterrows():
    print(f"=== SpearPhishMX - Fila {i} | label: {row['label']} ===")
    print(row['text'])
    print()
    print("-" * 100)
    print()

=== SpearPhishMX - Fila 2263 | label: 0 ===
Your account has been disabled ID : 0555027149 Apple Card | Goldman Sachs Estimado Jesús Cruz Vega: Como parte de nuestro Acuerdo de Seguridad hemos deshabilitado tu Apple ID. Hubo un problema con la información de la cuenta. Revisa toda la información personal y de seguridad en tu cuenta. Puedes reactivar tu cuenta siguiendo las instrucciones. Actualizar cuenta Si no recibimos noticias tuyas en un plazo de 48 horas, tu iCloud será suspendido y cualquier cuenta o información guardada se perderá. Pedimos disculpas por los inconvenientes y esperamos tener noticias tuyas pronto. Presentamos Apple Card Family Ahora puedes compartir la Apple Card con tu grupo de En Familia. Agrega a un compañero, enseña a los niños hábitos de gasto saludables o compártela con quien consideres tu familia. Apple Apple Resumen de Apple ID • Términos de venta • Política de privacidad Copyright © 2022 Apple Inc. Todos los derechos reservados

--------------------------

## 5. Exportación final

Se guardan como dos archivos separados, sin combinar ni balancear —
estos conjuntos se usan tal como son para la evaluación fuera de
distribución.

In [28]:
df_spaphish.to_csv('/content/drive/MyDrive/spaphish_limpio.csv', index=False, encoding='utf-8-sig')
df_spearphishmx.to_csv('/content/drive/MyDrive/spearphishmx_limpio.csv', index=False, encoding='utf-8-sig')

print('Guardados: spaphish_limpio.csv y spearphishmx_limpio.csv')

Guardados: spaphish_limpio.csv y spearphishmx_limpio.csv
